In [2]:
!wget -O telco_churn.csv https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv

--2026-08-14 18:42:33--  https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 970457 (948K) [text/plain]
Saving to: ‘telco_churn.csv’

telco_churn.csv     100%[===================>] 947.71K  5.73MB/s    in 0.2s    

2026-08-14 18:42:33 (5.73 MB/s) - ‘telco_churn.csv’ saved [970457/970457]



In [3]:
import pandas as pd

df = pd.read_csv("telco_churn.csv")
print(df.shape)
print(df["Churn"].value_counts(normalize=True))

(7043, 21)
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split

# Drop kolom customerID jika ada (opsional, karena bukan fitur prediktif)
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# 1. Konversi target variable 'Churn' menjadi biner (1 untuk Yes, 0 untuk No)
y = df['Churn'].map({'Yes': 1, 'No': 0}) if df['Churn'].dtype == 'object' else df['Churn']

# 2. Fitur (X) diambil dari DataFrame tanpa kolom 'Churn'
X = df.drop(columns=['Churn'])

# 3. Encoding fitur kategorikal menggunakan pd.get_dummies
X = pd.get_dummies(X, drop_first=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier( n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [6]:
from sklearn.metrics import classification_report, roc_auc_score

# 1. Hitung prediksi label dan probabilitas
y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

# 2. Tampilkan classification_report dan ROC-AUC
print("=== Classification Report ===")
print(classification_report(y_te, y_pred))

auc_score = roc_auc_score(y_te, y_proba)
print(f"ROC-AUC Score: {auc_score:.4f}")

# 3. Tampilkan sampel probabilitas churn
print("\nSampel Probabilitas Churn (5 data pertama):")
print(y_proba[:5])

# 4. Kesimpulan singkat
print("\n=== KESIMPULAN ===")
print(
    "1. Model Random Forest berhasil dilatih untuk memprediksi churn pelanggan "
    "dengan menangani ketidakseimbangan kelas menggunakan parameter class_weight='balanced'.\n"
    "2. Performa model dievaluasi secara menyeluruh menggunakan skor ROC-AUC dan Classification Report "
    "untuk mengukur akurasi, presisi, serta recall pada kelas pelanggan churn.\n"
    "3. Probabilitas churn yang dihasilkan dari predict_proba memungkinkan tim bisnis "
    "mengidentifikasi pelanggan berisiko tinggi dan mengambil tindakan retensi secara proaktif."
)

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8301

Sampel Probabilitas Churn (5 data pertama):
[0.00666667 0.65666667 0.05       0.3        0.02      ]

=== KESIMPULAN ===
1. Model Random Forest berhasil dilatih untuk memprediksi churn pelanggan dengan menangani ketidakseimbangan kelas menggunakan parameter class_weight='balanced'.
2. Performa model dievaluasi secara menyeluruh menggunakan skor ROC-AUC dan Classification Report untuk mengukur akurasi, presisi, serta recall pada kelas pelanggan churn.
3. Probabilitas churn yang dihasilkan dari predict_proba memungkinkan tim bisnis mengidentifikasi pelanggan berisiko tinggi dan mengambil tindakan retensi secara pro